# Understanding customer churn in retail banking

### Identify factors associated with customer churn and provide recommendations to improve retention.

***

**Notebook 1: Data Preparation**

1. The Business Problem
2. Load CSVs
3. Audit columns: customers table
4. Audit columns: activity table
5. Creating SQLite database and loading tables into SQL
6. Data cleaning in SQL: customers table
7. Data cleaning in SQL: activity table
8. Re-loading cleaned Dataframes to SQL
9. Customer snapshot
10. Create analysis_df.csv
11. Notes on datasets created

***
**<u>1. The Business Problem</u>**

Customer churn is one of the biggest real problems a bank is likely to have as retaining customers is usually much cheaper than acquring new ones.  Churn occurs when customers close current accounts, credit cards, loans or savings products.  

In this case study, I will be analysing the following to indicate which customers are most likely to leave:

- transaction frequency
- account balances
- product holdings
- customer tenure
- complaints
- digital banking usage

For the purposes of this case study, churned = YES means that the customer left the bank during the analysis period, and churned = NO means that the customer remained a customer.

See 11. Notes on datasets created for further information.

***

**<u>2. Load CSVs</u>**

In [32]:
import pandas as pd

customers = pd.read_csv("retail_bank_customers_v2_messy.csv")

activity = pd.read_csv("retail_bank_monthly_activity_v2_messy.csv")

***
**<u>3. Audit columns: customers table</u>**

In [33]:
customers.head()

,customer_id,age,tenure_months,region,income,number_of_products,complaint_count,churned
0,42416431,36.0,15,London,90300.0,2.0,0,No
1,90321228,50.0,102,England - North West,25900.0,3.0,0,No
2,78766521,35.0,36,England - South West,24900.0,2.0,0,No
3,66911344,66.0,73,England - London,17600.0,3.0,0,No
4,16182082,36.0,18,England - North West,33500.0,1.0,0,No


In [34]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         5030 non-null   int64  
 1   age                 4940 non-null   float64
 2   tenure_months       5030 non-null   int64  
 3   region              4968 non-null   object 
 4   income              4856 non-null   object 
 5   number_of_products  5000 non-null   float64
 6   complaint_count     5030 non-null   int64  
 7   churned             5030 non-null   object 
dtypes: float64(2), int64(3), object(3)
memory usage: 314.5+ KB


***
My process is to methodically go through columns one by one to understand the data and to do any necessary preliminary cleaning using pandas for data type/formatting problems.  I will later use SQL to clean data for business rules and standardisation.

***
**customer_id**

First I will check to see if there are any duplicate values in the customer_id column.

In [35]:
customers['customer_id'].duplicated().sum()

30

There are 30 duplicated records.  To investigate what kind of duplicates these are, I save the list of duplicated customer_id records to a duplicate_customers variable, so these can be accessed later:

In [36]:
duplicate_customers = (
    customers[
    customers['customer_id'].duplicated(keep=False)
    ].sort_values('customer_id')
)
duplicate_customers.head()

,customer_id,age,tenure_months,region,income,number_of_products,complaint_count,churned
405,11607465,61.0,86,west midlands,13400.0,2.0,0,No
2992,11607465,61.0,86,west midlands,13900.0,2.0,2,No
2770,14985339,42.0,13,England - London,69100.0,2.0,1,Yes
3244,14985339,42.0,13,England - London,69100.0,2.0,1,Yes
3926,16267462,29.0,21,England - South East,34800.0,2.0,0,NO


From the duplicate_customers DataFrame, I can see a mixture of exact duplicate records (for instance, customer 14985339) and records where there is an identical customer id, but different data (for instance, customer 11607465).  For the exact duplicate records, my approach will be to remove one copy.  I first want to check how many times a customer ID appears duplicated:

In [37]:
duplicate_customers.groupby("customer_id").size()

customer_id
11607465    2
14985339    2
16267462    2
26865646    2
28628135    2
29972694    2
30718182    2
32480786    2
34807166    2
36048167    2
37847837    2
40815287    2
44112124    2
46788825    2
56587926    2
65452259    2
66675691    2
67511606    2
69164873    2
69167577    2
75098742    2
81709934    2
82367526    2
88573640    2
89146418    2
89394433    2
91682856    2
96633717    2
98615058    2
98619306    2
dtype: int64

Before I delete records where there is an idential customer id but different data, I will first see how many of these records exist:

In [38]:
exact_duplicates = customers.duplicated().sum()

duplicate_customer_rows = (
    customers["customer_id"].duplicated().sum()
)

conflicting_duplicates = (
    duplicate_customer_rows
    - exact_duplicates
)

print(f"Exact duplicates: {exact_duplicates}")
print(f"Conflicting duplicates: {conflicting_duplicates}")

Exact duplicates: 17
Conflicting duplicates: 13


I will take a pragmatic approach and drop the exact duplicates but keep the first record for those conflicting duplicates:

In [39]:
customers = customers.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

Check the duplicate count now after running the above:

In [79]:
customers['customer_id'].duplicated().sum()

0

***
**Outcome**

30 duplicate customer records were identified.  Seventeen were exact duplicates and were removed.  Thirteen contained conflicting values, the first occurrence was retained to ensure a unique customer-level record for subsequent analysis.

***

**age**

In [42]:
customers['age'].describe()

count    4910.000000
mean       43.200815
std        14.611963
min        15.000000
25%        33.000000
50%        43.000000
75%        53.000000
max       120.000000
Name: age, dtype: float64

In [43]:
customers['age'].sort_values().head()

1929    15.0
3874    15.0
3113    16.0
4992    16.0
1739    18.0
Name: age, dtype: float64

In [46]:
customers['age'].dropna().sort_values().tail(10)

4600     88.0
4887     88.0
615      88.0
4564     88.0
2027     99.0
729     104.0
1771    104.0
771     104.0
13      120.0
395     120.0
Name: age, dtype: float64

From our initial interrogation of the data in the 'age' column, we can see that four customers are under the age of 18, and five customers over the age of 100.  For my churn analysis study, I am choosing to nullify the ages under 18 as most customers appear to be adults and our dataset does not contain any indicator that these are junior accounts.  I will set the NULL later in my SQL cleaning as I am purely using pandas to clean data types and formatting issues.  Setting these to NULL will mean that the record for the customer will still exist but pandas will ignore the entry when aggregating customer data.

I will also set any ages over 100 to NULL as I think it will be a reasonble business rule that valid ages are between 18 and 100.

***
**Outcome**

I identified nine records with ages outside of the plausible range of 18-100.  As age was not critical for retaining the customer record, I set these values to NULL and retained the customers for subsequent analysis.

***
**tenure_months**

In [48]:
customers['tenure_months'].describe()

count    5000.000000
mean       43.082000
std        27.969872
min         0.000000
25%        22.000000
50%        40.000000
75%        60.000000
max       155.000000
Name: tenure_months, dtype: float64

In [51]:
customers['tenure_months'].sort_values().head(20)

1237    0
1150    0
3860    0
1134    0
3882    0
1114    0
1113    0
3894    0
1103    0
3897    0
3902    0
1097    0
1083    0
3938    0
1155    0
3941    0
3954    0
1049    0
1030    0
3988    0
Name: tenure_months, dtype: int64

In [55]:
customers['tenure_months'].dropna().sort_values(ascending=False).head(10)

1655    155
589     155
4564    149
4994    148
2880    145
3080    144
4979    143
1838    139
2385    139
2024    139
Name: tenure_months, dtype: int64

In [78]:
(customers['tenure_months']== 0).sum()

287

I note that out of our 5,000 records (with duplicated customer ids removed), 287 customers have a tenure_months value of 0, which equates to 5.7%.  In a banking context, this could mean that 287 customers joined the bank in the month, which is plausible.  Later when I perform EDA, I will investigate whether brand new customers churn differently so we will keep these records in our dataset for now for further analysis.

***
**region**

In [59]:
customers['region'].value_counts(dropna=False)

region
England - London                    833
England - South East                661
England - North West                611
England - Yorkshire and Humber      497
England - West Midlands             489
England - South West                424
Scotland                            387
Wales                               264
England - North East                233
Northern Ireland                    171
NaN                                  60
London                               28
england - london                     27
 England - London                    21
England - NW                         18
south east                           18
England - SE                         18
Yorks & Humber                       15
 England - South East                15
north west                           15
 England - South West                15
west midlands                        14
 England - North West                13
 Scotland                            13
 England - Yorkshire and Humber  

Upon analysis of the region column, it is clear that regions will require standardisation into specific regions.  I will approach this later in my SQL cleaning step.  I will also designate NULL regions as 'Unknown' for easy analysis later.

***
**income**

After running our initial audit above, I can see that the data type for income is object.  This is suspicious so I need to sample a selection of rows.

In [61]:
customers["income"].dtype

dtype('O')

In [66]:
test_income=pd.to_numeric(
    customers['income'],
    errors='coerce'
)

In [67]:
customers.loc[test_income.isna(),"income"]

47            NaN
60            NaN
79            NaN
167           NaN
247           NaN
          ...    
4930     112,800 
4938      97,500 
4955          NaN
4994          NaN
5021      25,900 
Name: income, Length: 260, dtype: object

It looks like commas are preventing pandas from treating the income column as numeric.  As this is a data type issue, I will clean this column using pandas:

In [68]:
customers['income']=(
    customers['income']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)

In [70]:
customers['income']=pd.to_numeric(
    customers['income'],
    errors='coerce'
)

In [71]:
customers['income'].dtype

dtype('float64')

The income column now has a data type of float 64 which is what we wanted to achieve.

In [72]:
customers['income'].describe()

count      4826.000000
mean      41334.376295
std       30311.378212
min      -60200.000000
25%       26000.000000
50%       36200.000000
75%       50500.000000
max      750000.000000
Name: income, dtype: float64

In [73]:
customers['income'].sort_values().head(20)

4732   -60200.0
3678   -58600.0
2150   -47900.0
1047   -43600.0
4640   -41200.0
1633   -36800.0
2401   -32500.0
2265   -30300.0
4590   -28900.0
3334   -25000.0
1635   -22600.0
3677   -13100.0
1164    12000.0
2218    12000.0
2209    12000.0
4318    12000.0
3621    12000.0
4847    12000.0
1357    12000.0
4608    12000.0
Name: income, dtype: float64

In [76]:
customers['income'].dropna().sort_values(ascending=False).head(20)

797     750000.0
3844    750000.0
562     750000.0
4333    500000.0
3055    500000.0
4923    500000.0
4906    325000.0
3763    325000.0
1830    180000.0
2454    180000.0
4258    163500.0
3017    162500.0
559     160000.0
3807    157000.0
4934    153700.0
1818    148100.0
2612    145600.0
1518    145500.0
4434    145200.0
847     144900.0
Name: income, dtype: float64

In [77]:
(customers['income']<0).sum()

12

Upon analysis of our cleaned income column, we see that twelve customers have an income less than zero.  This could be due to a data entry error, a missing value coded incorrectly, or a genuine negative income.  I will replace these with NULL in my SQL cleaning step as I do not want to delete the customer records entirely as the remainder of the customer record may be perfectly useful.

***
**Outcome**

12 records contained negative annual income values.  As negative income was considered invalid for this analysis, these values were replaced with NULL values.

Values that were already missing were left as missing.

***
**number_of_products**

In [80]:
customers["number_of_products"].describe()

count    4970.000000
mean        2.013682
std         0.953652
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        12.000000
Name: number_of_products, dtype: float64

In [86]:
customers["number_of_products"].value_counts(dropna=False)

number_of_products
2.0     1975
1.0     1661
3.0      994
4.0      289
5.0       41
NaN       30
0.0        4
7.0        3
12.0       2
9.0        1
Name: count, dtype: int64

Here, we can see that two people hold 12 products and one person has 9.  We have to question whether this is plausible.  I will therefore create a business rule whereby we NULL values for a product count over 8.

In [87]:
customers.loc[customers['number_of_products']>8, 'number_of_products'].value_counts()

number_of_products
12.0    2
9.0     1
Name: count, dtype: int64

***
**Outcome**

I identified customers with a product count of 9 and 12.  Customers with more than 8 products were considered implausible based on the retail banking product set and were treated as missing values pending further business clarification.

***
**complaint_count**

In [89]:
customers["complaint_count"].value_counts(dropna=False)

complaint_count
0    3508
1    1205
2     239
3      40
4       7
5       1
Name: count, dtype: int64

***
**churned**

In [90]:
customers["churned"].value_counts(dropna=False)

churned
No       4475
Yes       400
no         31
NO         31
 No        30
N          23
YES         3
yes         3
 Yes        2
Y           2
Name: count, dtype: int64

Similar to the region column, I will seek to standardise the formatting of churned categories to return simple Yes and No values.  This will be done later in the SQL cleaning stage.

***
**<u>4. Audit columns: activity table</u>**

In [92]:
activity.head()

,customer_id,month,transaction_count,avg_balance,total_spend,mobile_logins,branch_visits
0,14473139,2025-11,36.0,4180.80,1694.27,11.0,1.0
1,84044453,2026-03,35.0,3067.72,887.85,21.0,0.0
2,30814469,2026-03,36.0,4767.73,681.57,21.0,0.0
3,68669864,2025-10,30.0,9671.12,2026.71,13.0,0.0
4,86255606,2025-12,34.0,3109.86,1244.86,30.0,0.0


In [93]:
activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50399 entries, 0 to 50398
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        50399 non-null  int64  
 1   month              50399 non-null  object 
 2   transaction_count  49997 non-null  float64
 3   avg_balance        49491 non-null  float64
 4   total_spend        49796 non-null  object 
 5   mobile_logins      49896 non-null  float64
 6   branch_visits      50096 non-null  float64
dtypes: float64(4), int64(1), object(2)
memory usage: 2.7+ MB


Duplicates in the customer_id column in the activity table are expected due to there being a one-to-many relationship with multiple rows per customer.  This does mean, however, that when we come to creating our customer snapshot, we will need to aggregate the data in the activity table.

***
**month**

For a churn analysis, I would want the month column to be in datetime format, so that I can sort chronologically, extract year/month, identify future dates and calculate periods.  Currently we can see that this column is in object format.

In [153]:
activity['month'].astype(str).sort_values().unique()

array(['2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01',
       '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01',
       '2026-02-01', '2026-03-01', '2026-04-01', '2026-05-01', 'NaT'],
      dtype=object)

Dates in the main are currently in the format YYYY-MM however we can see some anomalies here:

- leading/trailing spaces
- full dates like 2026-02-01 (valid dates, but we just want the month)
- text months like Jan-2026 (also valid information)
- impossible dates (2026-13)
- future dates

As these are data type/formatting issues and not a business rule problem, we will use pandas to clean.

In [96]:
activity['month'] = (
    activity['month']
    .astype(str)
    .str.strip()
    .str.replace('/', '-', regex=False)
)

We then convert to datetime

In [97]:
activity['month'] = pd.to_datetime(
    activity['month'],
    errors='coerce'
)

Here, we will check our output:

In [99]:
activity['month'].sort_values().unique()

<DatetimeArray>
['2025-06-01 00:00:00', '2025-07-01 00:00:00', '2025-08-01 00:00:00',
 '2025-09-01 00:00:00', '2025-10-01 00:00:00', '2025-11-01 00:00:00',
 '2025-12-01 00:00:00', '2026-01-01 00:00:00', '2026-02-01 00:00:00',
 '2026-03-01 00:00:00', '2026-04-01 00:00:00', '2026-05-01 00:00:00',
 '2027-01-01 00:00:00', '2028-04-01 00:00:00', '2099-12-01 00:00:00',
                 'NaT']
Length: 16, dtype: datetime64[ns]

Document the future dates:

In [100]:
activity.loc[activity['month']>"2026-05"]

,customer_id,month,transaction_count,avg_balance,total_spend,mobile_logins,branch_visits
3678,59693596,2099-12-01,31.0,1727.69,589.84,15.0,0.0
3695,97317034,2028-04-01,36.0,3940.28,"1,505.31",27.0,3.0
4338,71026436,2027-01-01,37.0,NaN,1326.59,20.0,1.0
6363,14420482,2028-04-01,28.0,2507.23,691.77,8.0,1.0
8531,84078997,2099-12-01,21.0,2397.85,604.21,14.0,3.0
14926,69167577,2028-04-01,30.0,2928.56,1240.93,16.0,0.0
15367,67677450,2099-12-01,45.0,3198.01,1822.81,28.0,0.0
15419,60440221,2099-12-01,20.0,7147.98,1982.45,15.0,2.0
17116,78233757,2027-01-01,52.0,6650.03,2372.59,28.0,0.0
21349,11955083,2028-04-01,41.0,4594.23,1192.98,23.0,2.0


In [101]:
(activity['month']>"2026-05").sum()

19

There are 19 future dated records, which we will set to missing:

In [102]:
activity.loc[activity['month']>"2026-05","month"]=pd.NaT

In [104]:
activity['month'].sort_values().unique()

<DatetimeArray>
['2025-06-01 00:00:00', '2025-07-01 00:00:00', '2025-08-01 00:00:00',
 '2025-09-01 00:00:00', '2025-10-01 00:00:00', '2025-11-01 00:00:00',
 '2025-12-01 00:00:00', '2026-01-01 00:00:00', '2026-02-01 00:00:00',
 '2026-03-01 00:00:00', '2026-04-01 00:00:00', '2026-05-01 00:00:00',
                 'NaT']
Length: 13, dtype: datetime64[ns]

***
**Outcome**

Monthly activity records were aggregated to customer level to create a customer snapshot dataset.  This ensured that each customer was represented by a single record prior to churn analysis.

***
**transaction_count**

In [106]:
activity["transaction_count"].describe()

count    49997.000000
mean        32.235934
std         14.391500
min          0.000000
25%         24.000000
50%         31.000000
75%         39.000000
max        750.000000
Name: transaction_count, dtype: float64

In [108]:
activity["transaction_count"].sort_values().head()

45647    0.0
29466    0.0
8855     1.0
1879     1.0
46248    2.0
Name: transaction_count, dtype: float64

In [110]:
activity["transaction_count"].sort_values(ascending=False).head(10)

23233    750.0
6036     750.0
30448    750.0
15286    750.0
25992    750.0
30497    750.0
1881     400.0
45417    400.0
29993    400.0
22149    400.0
Name: transaction_count, dtype: float64

I want to challenge whether 750 transactions per month (c. 25 transactions a day) is plausible.  It's not impossible when you take contactless payments, direct debits, standing orders and card transactions into consideration.  Seeing as though only six customers return this level of transaction count, I will leave them in my analysis.

***
**avg_balance**

In [111]:
activity["avg_balance"].describe()

count     49491.000000
mean       4204.822356
std        7386.070737
min      -12000.000000
25%        2587.710000
50%        3680.410000
75%        5115.700000
max      500000.000000
Name: avg_balance, dtype: float64

In [114]:
activity["avg_balance"].sort_values().head(15)

40892   -12000.00
44082   -12000.00
49026   -12000.00
250     -12000.00
31959   -12000.00
30171    -8500.00
28908    -8500.00
13778    -8500.00
4661     -8500.00
31555    -8500.00
45637     -999.79
11150     -861.47
14783     -801.04
15056     -615.35
36392     -600.64
Name: avg_balance, dtype: float64

In [115]:
activity["avg_balance"].dropna().sort_values(ascending=False).head(15)

20034    500000.0
43275    500000.0
7694     500000.0
30000    500000.0
10019    500000.0
48030    500000.0
2267     500000.0
32862    500000.0
11254    500000.0
2102     250000.0
15181    250000.0
24140    250000.0
13451    150000.0
44169    150000.0
8418     150000.0
Name: avg_balance, dtype: float64

Is it really possible to have an average balance of -£12k over the month?  As we have five customers with an avg_balance of -£12k, this seems a bit implausible as real-world values should surely vary.  As for customers on the other end of the scale, these could be genuinely wealthy customers.  I will therefore retain the high-balance customers but nullify the balances of -£12k.

***
**Outcome**

Five customers exhibited average balances of -£12,000.  These were deemed implausible relative to the retail banking population and treated as missing values.

High-balance values were investigated and determined to be plausible customer balances rather than data quality issues.  These records were retained.

***
**total_spend**

Per the initial auditing of the data in the activity table, we can see that total_spend data type is of object.  It's therefore necessary to see what is causing pandas to not recognise the column as numeric:

In [118]:
test_spend = pd.to_numeric(
    activity["total_spend"],
    errors="coerce"
)

In [119]:
activity.loc[test_spend.isna(),"total_spend"]

21              NaN
44              NaN
65              NaN
118             NaN
162       1,445.89 
            ...    
49898           NaN
50013     6,644.54 
50113     1,208.15 
50292           NaN
50304     3,382.46 
Name: total_spend, Length: 854, dtype: object

Similar to the income column in the customers table, it's the commas that are the issue here so we shall remove using pandas.

In [120]:
activity['total_spend'] = (
    activity['total_spend']
    .astype(str)
    .str.replace(',','', regex=False)
    .str.strip()
)

In [122]:
activity['total_spend'] = pd.to_numeric(
    activity['total_spend'],
    errors='coerce'
)

In [123]:
activity['total_spend'].dtype

dtype('float64')

In [124]:
activity['total_spend'].describe()

count     49796.000000
mean       1340.569400
std        1519.465854
min           0.000000
25%         628.607500
50%        1019.900000
75%        1676.892500
max      100000.000000
Name: total_spend, dtype: float64

In [128]:
activity['total_spend'].dropna().sort_values(ascending=False).head(15)

38817    100000.00
45908    100000.00
35900    100000.00
39109    100000.00
23868    100000.00
10883     50000.00
48177     50000.00
28905     50000.00
6043      25000.00
28760     25000.00
44480     25000.00
19997     11101.49
22505     10916.79
38875     10891.10
47176     10875.77
Name: total_spend, dtype: float64

***
**mobile_logins**

In [131]:
activity['mobile_logins'].describe()

count    49896.000000
mean        18.529401
std         11.488468
min          0.000000
25%         11.000000
50%         17.000000
75%         24.000000
max        500.000000
Name: mobile_logins, dtype: float64

In [132]:
activity['mobile_logins'].dropna().sort_values(ascending=False).head(15)

678      500.0
43177    500.0
50227    500.0
11213    500.0
6994     365.0
6609     365.0
22680    365.0
11530    365.0
4506     365.0
47508    365.0
8845     365.0
13222    240.0
29487    240.0
29030    240.0
10781    240.0
Name: mobile_logins, dtype: float64

There seem to be some extreme outliers although it's not entirely impossible for an individual to log in circa 16 times per day.  Therefore, I will retain these for the purposes of future churn analysis.

***
**Outcome**

I investigated 11 extreme mobile login outliers and retained them as genuine customer behaviour.

***
**branch_visits**

In [133]:
activity['branch_visits'].describe()

count    50096.000000
mean         0.482553
std          0.759131
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          7.000000
Name: branch_visits, dtype: float64

In [134]:
activity['branch_visits'].dropna().sort_values(ascending=False).head(15)

21955    7.0
11347    7.0
49583    6.0
24454    6.0
47283    6.0
15455    6.0
24216    6.0
3305     6.0
33171    6.0
39789    6.0
23140    6.0
9356     6.0
3354     6.0
12507    5.0
32607    5.0
Name: branch_visits, dtype: float64

I would assume that 7 branch visits per month would be pretty reasonable and I might perform some future analysis to see if there is a correlation between this and age, with the hypotheses that older customers may prefer in-branch visits to mobile app usage.

***
**<u>5. Creating SQLite database and loading tables into SQL</u>**

In [136]:
import sqlite3
conn = sqlite3.connect("bank.db")

In [137]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

activity.to_sql(
    "activity",
    conn,
    if_exists="replace",
    index=False
)


50399

***
**<u>6. Data cleaning in SQL: customers table</u>**

I have chosen to create one cleaned customer table rather than cleaning each column separately and storing lots of intermediate versions.  Here is my single SQL query that handles all the cleaning at once:

- age for customers outside of the 18-100 age bracket will be set to NULL
- regions will be standardised to the main regions of the UK.  NULL regions were designated as 'Unknown' for easy analysis later
- income for customers below zero will be set to NULL
- number of products for customers who hold over 8 products will be set to NULL
- churned column will return either a YES or NO value

In [154]:
query="""
SELECT
    customer_id,
    CASE
        WHEN age BETWEEN 18 AND 100 THEN age
        ELSE NULL
    END AS age,
    tenure_months,
    CASE
        WHEN LOWER(TRIM(region)) IN ('scotland', 'scot.')
            THEN 'Scotland'
        WHEN LOWER(TRIM(region)) IN ('wales')
            THEN 'Wales'
        WHEN LOWER(TRIM(region)) IN ('northern ireland', 'n. ireland')
            THEN 'Northern Ireland'
        WHEN LOWER(TRIM(region)) IN ('england - london', 'london')
            THEN 'England - London'
        WHEN LOWER(TRIM(region)) IN ('england - south east', 'south east', 'england - se')
            THEN 'England - South East'
        WHEN LOWER(TRIM(region)) IN ('england - north west', 'north west', 'england - nw')
            THEN 'England - North West'
        WHEN LOWER(TRIM(region)) IN ('england - west midlands', 'west midlands', 'west mids')
            THEN 'England - West Midlands'
        WHEN LOWER(TRIM(region)) IN ('england - south west', 'south west', 'england - sw')
            THEN 'England - South West'
        WHEN LOWER (TRIM(region)) IN ('england - north east', 'england - ne', 'north east')
            THEN 'England - North East'
        WHEN LOWER (TRIM(region)) IN ('england - yorkshire and humber', 'yorks & humber', 'yorkshire')
            THEN 'England - Yorkshire and Humber'
        WHEN region IS NULL THEN 'Unknown'
        ELSE TRIM(region)
    END AS region,
    CASE
        WHEN income < 0 THEN NULL
        ELSE income
    END AS income,
    CASE
        WHEN number_of_products > 8 THEN NULL
        ELSE number_of_products
    END AS number_of_products,
    complaint_count,
    CASE
        WHEN UPPER(TRIM(churned)) IN ('YES', 'Y')
            THEN 'YES'
        WHEN UPPER(TRIM(churned)) IN ('NO', 'N')
            THEN 'NO'
    END AS churned
FROM customers        
"""
cleaned_customers=pd.read_sql(query, conn)
cleaned_customers


,customer_id,age,tenure_months,region,income,number_of_products,complaint_count,churned
0,42416431,36.0,15,England - London,90300.0,2.0,0,NO
1,90321228,50.0,102,England - North West,25900.0,3.0,0,NO
2,78766521,35.0,36,England - South West,24900.0,2.0,0,NO
3,66911344,66.0,73,England - London,17600.0,3.0,0,NO
4,16182082,36.0,18,England - North West,33500.0,1.0,0,NO
...,...,...,...,...,...,...,...,...
4995,14018243,32.0,51,Scotland,55000.0,5.0,0,NO
4996,42982942,54.0,84,England - South East,14500.0,1.0,0,NO
4997,54414907,24.0,7,Wales,16000.0,1.0,3,YES
4998,86480066,34.0,42,England - North West,34200.0,2.0,0,NO


I will now perform some data checks to ensure data was cleaned properly.

In [139]:
cleaned_customers["age"].describe()

count    4901.000000
mean       43.154866
std        14.443072
min        18.000000
25%        33.000000
50%        43.000000
75%        53.000000
max        99.000000
Name: age, dtype: float64

Age is now as expected with the range being between 18 and 100.

In [155]:
cleaned_customers["region"].value_counts()

region
England - London                  909
England - South East              712
England - North West              657
England - Yorkshire and Humber    535
England - West Midlands           523
England - South West              452
Scotland                          425
Wales                             288
England - North East              259
Northern Ireland                  180
Unknown                            60
Name: count, dtype: int64

Regions are now standardised as expected.

In [141]:
cleaned_customers["income"].describe()

count      4814.000000
mean      41528.957208
std       30089.461055
min       12000.000000
25%       26100.000000
50%       36300.000000
75%       50500.000000
max      750000.000000
Name: income, dtype: float64

Negative values have now been omitted from our DataFrame.

In [142]:
cleaned_customers["number_of_products"].value_counts(dropna=False)

number_of_products
2.0    1975
1.0    1661
3.0     994
4.0     289
5.0      41
NaN      33
0.0       4
7.0       3
Name: count, dtype: int64

Product counts equal to and greater than 8 have now been removed.

In [143]:
cleaned_customers["churned"].value_counts(dropna=False)

churned
NO     4590
YES     410
Name: count, dtype: int64

Churned values are now standardised as expected.

***
**<u>7. Data cleaning in SQL: activity table</u>**

In our SQL data cleaning step for the activity table, the value of average balances if less than -£10k will be set to NULL as per justification above.

In [144]:
activity_query="""
SELECT
	customer_id,
	month,
	transaction_count,
	CASE 
		WHEN avg_balance <= -10000 THEN NULL
		ELSE avg_balance
	END AS avg_balance,
	total_spend,
	mobile_logins,
	branch_visits
FROM activity
"""
cleaned_activity=pd.read_sql(activity_query,conn)
cleaned_activity

,customer_id,month,transaction_count,avg_balance,total_spend,mobile_logins,branch_visits
0,14473139,2025-11-01 00:00:00,36.0,4180.80,1694.27,11.0,1.0
1,84044453,2026-03-01 00:00:00,35.0,3067.72,887.85,21.0,0.0
2,30814469,2026-03-01 00:00:00,36.0,4767.73,681.57,21.0,0.0
3,68669864,2025-10-01 00:00:00,30.0,9671.12,2026.71,13.0,0.0
4,86255606,2025-12-01 00:00:00,34.0,3109.86,1244.86,30.0,0.0
...,...,...,...,...,...,...,...
50394,13225719,2026-01-01 00:00:00,35.0,3259.18,546.21,31.0,1.0
50395,30507812,2026-03-01 00:00:00,29.0,5542.07,971.32,14.0,1.0
50396,50145591,2026-02-01 00:00:00,31.0,4838.44,914.74,19.0,0.0
50397,84535919,2025-07-01 00:00:00,27.0,3557.40,1337.62,12.0,0.0


In [145]:
cleaned_activity["avg_balance"].describe()

count     49486.00000
mean       4206.45967
std        7384.64742
min       -8500.00000
25%        2587.90750
50%        3680.99000
75%        5116.18750
max      500000.00000
Name: avg_balance, dtype: float64

avg_balance was the only column that we had to clean in SQL and we can see that the balances of -£12.5k have now been omitted.

***
**<u>8. Re-loading cleaned Dataframes to SQL</u>**

In [156]:
cleaned_customers.to_sql(
   "cleaned_customers",
   conn,
   if_exists="replace",
   index=False
)

cleaned_activity.to_sql(
   "cleaned_activity",
   conn,
   if_exists="replace",
   index=False
)

50399

***
**<u>9. Customer snapshot</u>**

An analysis dataset has been created from the cleaned activity and customer DataFrames.  Data in the cleaned_activity DataFrame had to be aggregated first for the result to return one row per customer.  This is because churn is a customer-level outcome.  Even though the month column in the activity DataFrame was cleaned, monthly activity records were aggregated to customer level to create a customer snapshot dataset.  This ensured that each customer was represented by a single record prior to churn analysis.

We also use a LEFT JOIN as we want to retain every customer and bring into the activity summary where available.  This means that new customers with no activity yet will stay in the dataset, with blank/NULL activity fields.

In [157]:
combined_query = """
WITH customer_snapshot AS(
    SELECT
        customer_id,
        SUM(transaction_count) AS total_transactions,
        AVG(avg_balance) AS avg_balance,
        SUM(total_spend) AS total_spend,
        SUM(mobile_logins) AS total_mobile_logins,
        SUM(branch_visits) AS total_branch_visits
    FROM cleaned_activity
    GROUP BY customer_id
    )
SELECT
    c.customer_id,
    c.age,
    c.tenure_months,
    c.region,
    c.income,
    c.number_of_products,
    c.complaint_count,
    c.churned,
    cs.total_transactions,
    cs.avg_balance,
    cs.total_spend,
    cs.total_mobile_logins,
    cs.total_branch_visits
FROM cleaned_customers AS c
LEFT JOIN customer_snapshot AS cs
ON c.customer_id = cs.customer_id
"""
analysis_df=pd.read_sql(combined_query,conn)
analysis_df


,customer_id,age,tenure_months,region,income,number_of_products,complaint_count,churned,total_transactions,avg_balance,total_spend,total_mobile_logins,total_branch_visits
0,42416431,36.0,15,England - London,90300.0,2.0,0,NO,424.0,51985.775000,33966.260000,222.0,3.0
1,90321228,50.0,102,England - North West,25900.0,3.0,0,NO,661.0,3685.040000,16226.977515,418.0,4.0
2,78766521,35.0,36,England - South West,24900.0,2.0,0,NO,348.0,3715.157500,6743.320000,209.0,4.0
3,66911344,66.0,73,England - London,17600.0,3.0,0,NO,348.0,780.408333,7812.720000,184.0,8.0
4,16182082,36.0,18,England - North West,33500.0,1.0,0,NO,334.0,4700.130833,9792.650000,203.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,14018243,32.0,51,Scotland,55000.0,5.0,0,NO,335.0,4771.128333,19436.000000,241.0,1.0
4996,42982942,54.0,84,England - South East,14500.0,1.0,0,NO,299.0,877.447500,5631.310000,152.0,2.0
4997,54414907,24.0,7,Wales,16000.0,1.0,3,YES,98.0,1380.457143,1964.180000,41.0,0.0
4998,86480066,34.0,42,England - North West,34200.0,2.0,0,NO,395.0,2291.115000,11179.110000,287.0,3.0


***
**<u>10. Create analysis_df.csv</u>**

In [162]:
analysis_df.to_csv(
    "analysis_df.csv",
    index=False
)

In [159]:
analysis_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          5000 non-null   int64  
 1   age                  4901 non-null   float64
 2   tenure_months        5000 non-null   int64  
 3   region               5000 non-null   object 
 4   income               4814 non-null   float64
 5   number_of_products   4967 non-null   float64
 6   complaint_count      5000 non-null   int64  
 7   churned              5000 non-null   object 
 8   total_transactions   4700 non-null   float64
 9   avg_balance          4697 non-null   float64
 10  total_spend          4698 non-null   float64
 11  total_mobile_logins  4700 non-null   float64
 12  total_branch_visits  4700 non-null   float64
dtypes: float64(8), int64(3), object(2)
memory usage: 507.9+ KB


In [160]:
analysis_df.describe()

,customer_id,age,tenure_months,income,number_of_products,complaint_count,total_transactions,avg_balance,total_spend,total_mobile_logins,total_branch_visits
count,5.000000e+03,4901.000000,5000.000000,4814.000000,4967.000000,5000.000000,4700.000000,4697.000000,4698.000000,4700.000000,4700.000000
mean,5.494668e+07,43.154866,43.082000,41528.957208,2.008254,0.367200,342.671064,4172.139902,14195.482419,196.449362,5.123191
std,2.606757e+07,14.443072,27.969872,30089.461055,0.927351,0.629955,142.169470,3108.300429,11243.760046,96.543067,4.285206
min,1.004668e+07,18.000000,0.000000,12000.000000,0.000000,0.000000,2.000000,-542.040000,0.000000,0.000000,0.000000
25%,3.225066e+07,33.000000,22.000000,26100.000000,1.000000,0.000000,246.000000,2580.610833,7090.984816,128.750000,2.000000
50%,5.468418e+07,43.000000,40.000000,36300.000000,2.000000,0.000000,346.000000,3634.275000,11314.685000,190.000000,4.000000
75%,7.774757e+07,53.000000,60.000000,50500.000000,3.000000,1.000000,434.000000,5054.970000,17854.782500,254.000000,7.000000
max,9.997825e+07,99.000000,155.000000,750000.000000,7.000000,5.000000,1259.000000,88408.371667,131321.390000,724.000000,31.000000


In [161]:
analysis_df.isna().sum()

customer_id              0
age                     99
tenure_months            0
region                   0
income                 186
number_of_products      33
complaint_count          0
churned                  0
total_transactions     300
avg_balance            303
total_spend            302
total_mobile_logins    300
total_branch_visits    300
dtype: int64

This concludes the data preparation step as we now move onto the exploratory data analysis.

***
**<u>11. Notes on datasets created</u>**

For this case study, I asked ChatGPT to generate synthetic banking data in the form of a customers table and activity table.  A few realistic touches were included:

- 300 customers have no activity yet
- churn rate is about 8.2%
- activity covers June 2025 to May 2026
- uneven region/customer distributions
- churn is influenced by tenure, products, complaints, income and activity patterns

The data was:

- unevenly distributed (lots of middle earners, fewer high earners)
- slighly influenced by region (London and South East tend to be higher)
- somewhat influenced by age
- capped at realistic levels (£12k - £180k)

Income here relates to customer's annual gross salary/income in GBP  Current annual income is often not available for every customer in a real bank so should be seen as an estimate (obtained through customer records and product applications) rather than simply 'income'.